# Monte Carlo production & inventory planner (multi‑product)

This notebook builds a 10‑year Monte Carlo model for seed production and inventory,
using archetype/maturity inputs, yield & conversion uncertainty, and portfolio‑level
aggregation across multiple products.


**Imports and file path**

This block imports core Python and Plotly libraries, configures the default renderer for Colab, and sets the path to the Excel workbook that contains all input tables for the simulation.



In [124]:
import pandas as pd
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# ── File paths ────────────────────────────────────────────────────────────────
file_path      = "/content/data.xlsx"             # change if needed
sales_var_path = "/content/salesVariability.xlsx"  # change if needed


**Load sheets and basic cleaning**

This block loads the key Excel sheets (conversion rates, production yields, product parameters, sales volume parameters), standardizes their column names, and builds a clean Parent0→Archetype mapping used to join other datasets later.



In [125]:
# Load sheets
conv_tab   = pd.read_excel(file_path, sheet_name="Conversion rates")
yield_tab  = pd.read_excel(file_path, sheet_name="Production yields")
params_tab = pd.read_excel(file_path, sheet_name="Product parameters")
sales_raw  = pd.read_excel(file_path, sheet_name="Sales volume parameters", header=None)

# Clean column names
for df_ in [conv_tab, yield_tab, params_tab]:
    df_.columns = df_.columns.astype(str).str.strip()

# Validate and normalize Parent0 → Archetype mapping
for col in ["Parent0", "Archetype"]:
    if col not in params_tab.columns:
        raise ValueError("Product parameters must contain columns: Parent0, Archetype")

params_tab["Parent0"]  = params_tab["Parent0"].astype(str).str.strip()
params_tab["Archetype"] = params_tab["Archetype"].astype(str).str.strip()
parent_to_arch = params_tab[["Parent0", "Archetype"]].dropna()


**Sales variability parameters (lognormal)**

Loads `salesVariability.xlsx` and parses four parameter tables:

- **First-year sales mu / sigma²** — lognormal parameters for Year 1 sales volume
- **Growth rate mu / sigma²** — lognormal parameters for the year-over-year sales
  *multiplier* (per Patrick's note: the distribution describes the multiplier directly,
  e.g. 1.5 means 50% growth; subtract 1 to get the growth rate).

During simulation, `draw_sales_curve()` is called once per run to produce a unique
stochastic 10-year sales trajectory.  The median run equals `exp(mu)` by definition,
matching the historical median from `data.xlsx`.


In [126]:
# ── Parse salesVariability.xlsx ──────────────────────────────────────────────
sv_raw = pd.read_excel(sales_var_path,
                       sheet_name="Sales volume parameters", header=None)

# ── First-year sales mu  (rows 5-9, cols 0-4) ────────────────────────────────
sv_fy_mu = {}
for r in range(5, 10):
    arch = str(sv_raw.iat[r, 0]).strip()
    for mat, col in {85: 1, 95: 2, 105: 3, 115: 4}.items():
        sv_fy_mu[(arch, mat)] = float(sv_raw.iat[r, col])

# ── First-year sales sigma²  (rows 15-19, cols 0-4) ─────────────────────────
sv_fy_sig2 = {}
for r in range(15, 20):
    arch = str(sv_raw.iat[r, 0]).strip()
    for mat, col in {85: 1, 95: 2, 105: 3, 115: 4}.items():
        sv_fy_sig2[(arch, mat)] = float(sv_raw.iat[r, col])

# ── Growth rate mu  (rows 5-24, cols 7-15  →  years 2-10) ───────────────────
sv_gr_mu = {}
for r in range(5, 25):
    arch = sv_raw.iat[r, 7]
    mat  = sv_raw.iat[r, 8]
    if pd.isna(arch) or pd.isna(mat):
        continue
    arch = str(arch).strip()
    mat  = int(float(mat))
    for yr in range(2, 11):
        sv_gr_mu[(arch, mat, yr)] = float(sv_raw.iat[r, 9 + (yr - 2)])

# ── Growth rate sigma²  (rows 5-24, cols 26-34  →  years 2-10) ──────────────
sv_gr_sig2 = {}
for r in range(5, 25):
    arch = sv_raw.iat[r, 24]
    mat  = sv_raw.iat[r, 25]
    if pd.isna(arch) or pd.isna(mat):
        continue
    arch = str(arch).strip()
    mat  = int(float(mat))
    for yr in range(2, 11):
        sv_gr_sig2[(arch, mat, yr)] = float(sv_raw.iat[r, 26 + (yr - 2)])

archetypes_sv = sorted(set(k[0] for k in sv_fy_mu))
maturities_sv = sorted(set(k[1] for k in sv_fy_mu))
print("Sales variability loaded.")
print(f"  Archetypes : {archetypes_sv}")
print(f"  Maturities : {maturities_sv}")
print(f"  FY mu entries    : {len(sv_fy_mu)}   "
      f"FY sigma2 entries : {len(sv_fy_sig2)}")
print(f"  GR mu entries    : {len(sv_gr_mu)}  "
      f"GR sigma2 entries : {len(sv_gr_sig2)}")


Sales variability loaded.
  Archetypes : ['Bayer - Above ground', 'Bayer - Above/below ground', 'Conventional', 'Syngenta - above ground', 'Syngenta - above/below ground']
  Maturities : [85, 95, 105, 115]
  FY mu entries    : 20   FY sigma2 entries : 20
  GR mu entries    : 180  GR sigma2 entries : 180


**Yield and conversion prep**

This block prepares agronomic and processing uncertainty inputs by computing yield factors (actual vs planned), cleaning numeric fields, and joining yields and conversion rates to archetypes so we can later sample realistic yield and conversion variability by product.



In [127]:
# Yield: compute Yield_Factor = Actual / Planned
for col in ["Parent0", "Planned yield (bu/ac)", "Actual yield"]:
    if col not in yield_tab.columns:
        raise ValueError(f"Production yields sheet missing column: {col}")

yield_tab["Parent0"] = yield_tab["Parent0"].astype(str).str.strip()
yield_tab["Planned yield (bu/ac)"] = pd.to_numeric(yield_tab["Planned yield (bu/ac)"], errors="coerce")
yield_tab["Actual yield"]          = pd.to_numeric(yield_tab["Actual yield"], errors="coerce")

yield_tab["Yield_Factor"] = yield_tab["Actual yield"] / yield_tab["Planned yield (bu/ac)"]
yield_tab["Yield_Factor"] = yield_tab["Yield_Factor"].replace([np.inf, -np.inf], np.nan)

# Map archetype onto yield rows
yield_w_arch = yield_tab.merge(parent_to_arch, on="Parent0", how="left")

# Conversion table
for col in ["Parent0", "totalConversionRate"]:
    if col not in conv_tab.columns:
        raise ValueError(f"Conversion rates sheet missing column: {col}")

conv_tab["Parent0"] = conv_tab["Parent0"].astype(str).str.strip()
conv_tab["totalConversionRate"] = pd.to_numeric(conv_tab["totalConversionRate"], errors="coerce")

conv_w_arch = conv_tab.merge(parent_to_arch, on="Parent0", how="left")


**Sales‑sheet helper functions**

This block defines helper utilities to work with the messy sales‑parameter sheet: functions to normalize text, locate header cells, scan header ranges, clean archetype labels, convert year columns and percentages, and ignore non‑data rows.



In [128]:
def norm_txt(x):
    if pd.isna(x):
        return ""
    return str(x).strip().lower()

def find_cell(text):
    target = text.strip().lower()
    for r in range(sales_raw.shape[0]):
        for c in range(sales_raw.shape[1]):
            if target in norm_txt(sales_raw.iat[r, c]):
                return r, c
    return None

def scan_row_until_blank(r, start_c):
    end = start_c
    while (end < sales_raw.shape[1] and
           not pd.isna(sales_raw.iat[r, end]) and
           str(sales_raw.iat[r, end]).strip() != ""):
        end += 1
    return end

def clean_series(s):
    s = s.astype(str).str.strip()
    s = s.replace({"nan": np.nan, "None": np.nan, "": np.nan})
    return s

def is_bad_archetype(x):
    t = norm_txt(x)
    if t in ["", "archetype", "maturity"]:
        return True
    bad = [
        "median first year sales volumes",
        "average first year sales volumes",
        "median growth rates",
        "average growth rates",
        "relative sales year",
    ]
    return any(b in t for b in bad)

def normalize_year_cols(cols):
    m = {}
    for c in cols:
        try:
            m[int(float(str(c).strip()))] = c
        except Exception:
            pass
    return m

def to_rate(x):
    if pd.isna(x):
        return 0.0
    if isinstance(x, str):
        s = x.strip()
        if s.endswith("%"):
            s = s[:-1].strip()
            v = pd.to_numeric(s, errors="coerce")
            return 0.0 if pd.isna(v) else float(v) / 100.0
        v = pd.to_numeric(s, errors="coerce")
        if pd.isna(v):
            return 0.0
        v = float(v)
    else:
        v = float(x)
    return v / 100.0 if abs(v) > 2 else v


**Median first‑year sales block**

This block locates the “Median first year sales volumes” section in the sales sheet, extracts a clean Archetype × Maturity table, and builds median_sales_df, which stores baseline first‑year sales volumes for each maturity bucket.



In [129]:
# Locate "Median first year sales volumes" section
anchor = find_cell("Median first year sales volumes")
if anchor is None:
    raise ValueError("Couldn't find 'Median first year sales volumes' block.")
anchor_r, _ = anchor

median_header_r = None
for r in range(anchor_r, min(anchor_r + 30, sales_raw.shape[0])):
    if norm_txt(sales_raw.iat[r, 0]) == "archetype":
        median_header_r = r
        break
if median_header_r is None:
    raise ValueError("Couldn't find 'Archetype' header for Median first year sales block.")

median_start_c = 0
median_end_c = scan_row_until_blank(median_header_r, median_start_c)

median_df = sales_raw.iloc[median_header_r + 1:, median_start_c:median_end_c].copy()
median_df.columns = sales_raw.iloc[median_header_r, median_start_c:median_end_c].values

median_df = median_df.dropna(subset=["Archetype"])
median_df["Archetype"] = clean_series(median_df["Archetype"])
median_df = median_df[~median_df["Archetype"].apply(is_bad_archetype)]

# Map maturity columns
maturity_cols_map = {}
for col in median_df.columns:
    if norm_txt(col) == "archetype":
        continue
    try:
        maturity_cols_map[int(float(str(col).strip()))] = col
    except Exception:
        pass

needed_maturities = [85, 95, 105, 115]
missing = [m for m in needed_maturities if m not in maturity_cols_map]
if missing:
    raise ValueError(
        f"Missing maturity cols {missing} in median sales block. "
        f"Found: {sorted(maturity_cols_map.keys())}"
    )

median_sales_df = median_df[["Archetype"] + [maturity_cols_map[m] for m in needed_maturities]].copy()
median_sales_df.columns = ["Archetype"] + needed_maturities
for m in needed_maturities:
    median_sales_df[m] = pd.to_numeric(median_sales_df[m], errors="coerce")
median_sales_df = median_sales_df.dropna(subset=needed_maturities, how="all")


**Median growth rates block**

This block finds and parses the “Median growth rates” section, cleans Archetype and Maturity, maps generic Year2–Year10 columns, and validates that we have growth‑rate data for all required years used in the sales lifecycle.



In [130]:
growth_anchor = find_cell("Median growth rates")
if growth_anchor is None:
    raise ValueError("Couldn't find 'Median growth rates' block.")
ga_r, _ = growth_anchor

growth_header_r = None
growth_start_c = None
for r in range(ga_r, min(ga_r + 30, sales_raw.shape[0])):
    for c in range(sales_raw.shape[1] - 1):
        if norm_txt(sales_raw.iat[r, c]) == "archetype" and norm_txt(sales_raw.iat[r, c + 1]) == "maturity":
            growth_header_r = r
            growth_start_c = c
            break
    if growth_header_r is not None:
        break
if growth_header_r is None:
    raise ValueError("Couldn't find Archetype+Maturity header in Median growth rates block.")

growth_end_c = scan_row_until_blank(growth_header_r, growth_start_c)

growth_df = sales_raw.iloc[growth_header_r + 1:, growth_start_c:growth_end_c].copy()
growth_df.columns = sales_raw.iloc[growth_header_r, growth_start_c:growth_end_c].values

growth_df = growth_df.dropna(subset=["Archetype", "Maturity"])
growth_df["Archetype"] = clean_series(growth_df["Archetype"])
growth_df["Maturity"]  = pd.to_numeric(growth_df["Maturity"], errors="coerce")
growth_df = growth_df[~growth_df["Archetype"].apply(is_bad_archetype)]
growth_df = growth_df.dropna(subset=["Archetype", "Maturity"])

year_map = normalize_year_cols(
    [c for c in growth_df.columns if norm_txt(c) not in ["archetype", "maturity"]]
)

years_needed = list(range(2, 11))  # Year2..Year10
missing_years = [y for y in years_needed if y not in year_map]
if missing_years:
    raise ValueError(
        f"Missing growth year columns {missing_years} in Median growth rates block. "
        f"Found: {sorted(year_map.keys())}"
    )


**Lookup functions and sales curve**

This block provides lookup helpers for median first‑year sales and year‑over‑year growth rates, derives yield and conversion distributions by archetype, and builds a 10‑year expected sales curve per Archetype × Maturity that serves as the deterministic base for the Monte Carlo.



In [131]:
def get_median_sales(archetype, maturity):
    """Fallback: median first-year sales from data.xlsx."""
    row = median_sales_df[median_sales_df["Archetype"] == archetype]
    if row.empty:
        return None
    v = row[maturity].dropna()
    return None if v.empty else float(v.iloc[0])

def get_yoy_rates(archetype, maturity):
    """Fallback: median YoY growth rates from data.xlsx."""
    row = growth_df[
        (growth_df["Archetype"] == archetype) &
        (growth_df["Maturity"]  == maturity)
    ]
    if row.empty:
        return None
    return [to_rate(row[year_map[y]].iloc[0]) for y in years_needed]

def get_mean_std_by_archetype(archetype):
    y = yield_w_arch.loc[yield_w_arch["Archetype"] == archetype,
                         "Yield_Factor"].dropna()
    c = conv_w_arch.loc[conv_w_arch["Archetype"] == archetype,
                        "totalConversionRate"].dropna()
    y_mean = float(y.mean()) if len(y) else float(
        yield_w_arch["Yield_Factor"].dropna().mean())
    c_mean = float(c.mean()) if len(c) else float(
        conv_w_arch["totalConversionRate"].dropna().mean())
    y_std  = float(y.std(ddof=1)) if len(y) > 1 else 1e-6
    c_std  = float(c.std(ddof=1)) if len(c) > 1 else 1e-6
    if y_std <= 0: y_std = 1e-6
    if c_std <= 0: c_std = 1e-6
    return y_mean, y_std, c_mean, c_std

def sample_yield_conv_normal(archetype, rng):
    y_mean, y_std, c_mean, c_std = get_mean_std_by_archetype(archetype)
    return (float(max(0.0, rng.normal(y_mean, y_std))),
            float(max(0.0, rng.normal(c_mean, c_std))))

# ── Deterministic fallback sales curve (median values) ───────────────────────
def build_sales_curve(archetype, maturity):
    """Returns fixed 10-year sales array using median values. Fallback only."""
    y1  = get_median_sales(archetype, maturity)
    yoy = get_yoy_rates(archetype, maturity)
    if y1 is None or yoy is None:
        return None
    sales = [y1]
    for rate in yoy:
        sales.append(max(0.0, sales[-1] * (1 + rate)))
    return np.array(sales, dtype=float)

# ── Stochastic sales curve — called once per MC run ──────────────────────────
def draw_sales_curve(archetype, maturity, rng):
    """
    Draw one 10-year sales trajectory from lognormal parameters.

    Per Patrick's note: the growth rate distributions describe the year-over-year
    sales MULTIPLIER (e.g. 1.5 = 50% growth).  The draw is applied directly as a
    multiplier — no subtraction needed for the compounding step.

    Sentinel values near -9.21 in gr_mu → exp(-9.21) ≈ 0.0001, naturally
    collapsing sales at end of lifecycle.

    Falls back to build_sales_curve() if any parameter is missing.
    """
    fy_mu   = sv_fy_mu.get((archetype, maturity))
    fy_sig2 = sv_fy_sig2.get((archetype, maturity))

    if fy_mu is None or fy_sig2 is None:
        return build_sales_curve(archetype, maturity)

    # Draw Year 1 sales
    y1 = float(rng.lognormal(fy_mu, np.sqrt(max(fy_sig2, 1e-10))))
    sales = [y1]

    for yr in range(2, 11):
        gr_mu   = sv_gr_mu.get((archetype, maturity, yr))
        gr_sig2 = sv_gr_sig2.get((archetype, maturity, yr))

        if gr_mu is None or gr_sig2 is None:
            # Fallback: median growth rate for this year
            yoy = get_yoy_rates(archetype, maturity)
            rate = yoy[yr - 2] if yoy else 0.0
            sales.append(max(0.0, sales[-1] * (1.0 + rate)))
            continue

        # Draw multiplier directly — per Patrick: distribution is the multiplier
        multiplier = float(rng.lognormal(gr_mu, np.sqrt(max(gr_sig2, 1e-10))))
        sales.append(max(0.0, sales[-1] * multiplier))

    return np.array(sales, dtype=float)

# ── Validation helper ─────────────────────────────────────────────────────────
def validate_sales_curve(archetype, maturity, n=5000, seed=42):
    """
    Sanity check: compare simulated median/mean against data.xlsx medians.
    Call this after loading to verify the lognormal implementation is correct.

    Expected: simulated median ≈ data.xlsx median (within ~2%)
              simulated mean > median due to right skew (this is correct)
    """
    rng = np.random.default_rng(seed)
    curves = np.array([draw_sales_curve(archetype, maturity, rng)
                       for _ in range(n)])

    median_curve = np.median(curves, axis=0)
    mean_curve   = curves.mean(axis=0)
    ref_curve    = build_sales_curve(archetype, maturity)

    year_labels = [f"Year {i}" for i in range(1, 11)]
    df = pd.DataFrame({
        "data.xlsx median (reference)": ref_curve.round(1),
        "Simulated median":             median_curve.round(1),
        "Simulated mean":               mean_curve.round(1),
        "Median match (%)":             (np.abs(median_curve - ref_curve)
                                         / np.maximum(ref_curve, 1) * 100).round(1),
    }, index=year_labels)

    print(f"\n=== Sales curve validation: {archetype} | Maturity {maturity} ===")
    print(f"    (n={n:,} simulations, seed={seed})")
    print(f"    If implementation is correct, 'Simulated median' should closely")
    print(f"    match 'data.xlsx median'. Mean > median is expected (right skew).\n")
    display(df)
    return df

# ── Quick usage: uncomment to run validation ─────────────────────────────────
# validate_sales_curve("Bayer - Above ground", 85)
# validate_sales_curve("Conventional", 95)


In [132]:
def build_launch_cohorts():
    """
    From launch_plan_df, create a list of cohorts:
      (archetype, maturity, launch_year, n_products)
    """
    cohorts = []
    for (arch, mat) in launch_plan_df.index:
        for ycol in launch_plan_df.columns:
            n = int(launch_plan_df.loc[(arch, mat), ycol])
            if n <= 0:
                continue
            launch_year = int(ycol.split("_")[1])
            cohorts.append((arch, int(mat), launch_year, n))
    return cohorts


**Widgets (archetype, maturity, strategy, multipliers, rules, simulation)**

This block defines the interactive controls for the dashboard: a multi‑select list of products (Archetype | Maturity), production strategy and per‑year multipliers, simulation settings (iterations and seed), analysis‑year and threshold selectors, and knobs for carryover stop‑rules and minimum production floors.



In [133]:
# ==== PRODUCT SELECTORS (multi-product) ====
product_options = []
for a in sorted(median_sales_df["Archetype"].dropna().unique()):
    for m in [85, 95, 105, 115]:
        y1 = get_median_sales(a, m)
        if y1 is not None and not np.isnan(y1):
            product_options.append(f"{a} | {m}")

products_ms = widgets.SelectMultiple(
    options=product_options,
    value=(product_options[0],),
    description="Products:",
    layout=widgets.Layout(width="600px", height="140px")
)

# ==== PRODUCTION STRATEGY ====
strategy_dd = widgets.Dropdown(
    options=[
        ("Custom multiplier by year (Y1-Y10)", "custom"),
        ("Just-in-time 1.0x — produce exactly next year sales", "jit"),
        ("Conservative 1.2x — small safety buffer", "cons"),
        ("Aggressive 2.0x — large safety buffer", "aggr"),
    ],
    value="custom",
    description="Strategy:",
    layout=widgets.Layout(width="520px")
)

# 10 separate sliders, one for each year
year_mult_sliders = [
    widgets.FloatSlider(
        description=f"Y{i}:",
        min=0.0, max=5.0, step=0.1, value=1.5,
        style={"description_width": "30px"},
        layout=widgets.Layout(width="320px")
    )
    for i in range(1, 11)
]

def get_yearly_multipliers():
    strat = strategy_dd.value
    if strat == "custom":
        return [float(s.value) for s in year_mult_sliders]
    elif strat == "jit":   return [1.0] * 10
    elif strat == "cons":  return [1.2] * 10
    elif strat == "aggr":  return [2.0] * 10
    return [1.5] * 10

# ==== SIMULATION SETTINGS ====
iters_slider = widgets.IntSlider(
    description="Simulations:",
    min=100, max=5000, step=100, value=1000,
    layout=widgets.Layout(width="350px")
)
seed_box = widgets.IntText(
    value=42,
    description="Random seed:",
    layout=widgets.Layout(width="220px")
)

# ==== ANALYSIS YEAR + THRESHOLD ====
year_mode_dd = widgets.Dropdown(
    options=[
        ("Last Year of Sales — auto-detect lifecycle end", "last_sales"),
        ("Choose specific year", "custom")
    ],
    value="last_sales",
    description="Year mode:",
    layout=widgets.Layout(width="420px")
)
analysis_year_dd = widgets.Dropdown(
    options=[(f"Year {i}", i-1) for i in range(1, 11)],
    value=5-1,
    description="Analysis year:",
    layout=widgets.Layout(width="250px")
)
threshold_slider = widgets.FloatSlider(
    description="Carryover threshold:",
    min=0, max=50000, step=500, value=0.0,
    style={"description_width": "160px"},
    readout_format=".0f",
    layout=widgets.Layout(width="540px")
)

# ==== PRODUCTION CONSTRAINTS ====

# 1. Minimum production floor
# When enabled: if the strategy says to produce in a given year,
# production will never fall below this floor value.
# Does NOT force production in years where strategy plans zero.
enable_min_floor_chk = widgets.Checkbox(
    value=False,
    description="Enable minimum production floor",
    indent=False,
    layout=widgets.Layout(width="300px")
)
min_floor_box = widgets.BoundedFloatText(
    value=1000.0,
    min=0, max=500000, step=100,
    description="Floor (units):",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="260px")
)
min_floor_box.layout.visibility = "hidden"

def _toggle_floor(change):
    min_floor_box.layout.visibility = "visible" if enable_min_floor_chk.value else "hidden"
enable_min_floor_chk.observe(_toggle_floor, names="value")

# 2. Maximum carryover inventory
# When enabled: if carry-in inventory at the start of a year already
# exceeds this cap, production is skipped for that year — there is
# already enough stock to cover expected demand.
enable_max_carry_chk = widgets.Checkbox(
    value=False,
    description="Enable maximum carryover cap",
    indent=False,
    layout=widgets.Layout(width="300px")
)
max_carry_box = widgets.BoundedFloatText(
    value=10000.0,
    min=0, max=500000, step=500,
    description="Max carry-in (units):",
    style={"description_width": "140px"},
    layout=widgets.Layout(width="300px")
)
max_carry_box.layout.visibility = "hidden"

def _toggle_carry(change):
    max_carry_box.layout.visibility = "visible" if enable_max_carry_chk.value else "hidden"
enable_max_carry_chk.observe(_toggle_carry, names="value")


### Launch cohorts table (multi‑year)

This block defines an archetype × launch‑year table where each cell is the
number of products of that archetype launched in that calendar year. This
implements Patrick’s “input table” idea for modeling overlapping launch
cohorts across multiple years.


In [134]:
# ==== MULTI-YEAR LAUNCH PLAN (by archetype × maturity) ====

LAUNCH_YEARS = list(range(0, 6))  # 0..5
MATURITIES   = [85, 95, 105, 115]

# All Archetype × Maturity combos that exist in data
archs = sorted(median_sales_df["Archetype"].dropna().unique())
launch_index = [(a, m) for a in archs for m in MATURITIES]

launch_plan_df = pd.DataFrame(
    0,
    index=pd.MultiIndex.from_tuples(launch_index, names=["Archetype", "Maturity"]),
    columns=[f"LaunchYear_{y}" for y in LAUNCH_YEARS],
)

launch_widgets = {}
rows = []

for arch, mat in launch_index:
    year_widgets = []
    for y in LAUNCH_YEARS:
        w = widgets.BoundedIntText(
            value=int(launch_plan_df.loc[(arch, mat), f"LaunchYear_{y}"]),
            min=0,
            max=50,
            layout=widgets.Layout(width="60px")
        )
        year_widgets.append(w)
    launch_widgets[(arch, mat)] = year_widgets

    label_html = widgets.HTML(f"<b>{arch} | {mat}</b>&nbsp;&nbsp;")
    row_box = widgets.HBox([label_html] + year_widgets)
    rows.append(row_box)

header_labels = [widgets.HTML("&nbsp;")]
for y in LAUNCH_YEARS:
    header_labels.append(widgets.HTML(f"<b>Y{y}</b>"))
header_row = widgets.HBox(header_labels)

launch_plan_box = widgets.VBox([header_row] + rows)

def sync_launch_plan_df():
    for (arch, mat) in launch_index:
        for i, y in enumerate(LAUNCH_YEARS):
            launch_plan_df.loc[(arch, mat), f"LaunchYear_{y}"] = int(
                launch_widgets[(arch, mat)][i].value
            )

In [135]:
multi_year_box = widgets.VBox([
    widgets.HTML("<b>Multi-year launch plan</b>"),
    widgets.HTML(
        "<small>Each cell = number of new products you plan to launch "
        "for that Archetype × Maturity in that calendar year "
        "(Y0 = first year of the simulation).</small>"
    ),
    launch_plan_box,
])
multi_year_box.layout.display = "none"

In [136]:
def show_launch_plan():
    display(launch_plan_df)

# Call show_launch_plan() if you want to preview after edits

**Strategy multipliers and one‑run simulation with carryover rule**

This block aggregates the simulation across all selected products, averaging and summing their lifecycles, then computes the inventory‑risk summary metrics (mean, median, P90, and probabilities above zero and above the user‑defined remaining‑inventory threshold) for both the selected analysis year and end of lifecycle.


In [137]:
def get_yearly_multipliers():
    strat = strategy_dd.value
    if strat == "custom":
        return [float(s.value) for s in year_mult_sliders]
    elif strat == "jit":   return [1.0] * 10
    elif strat == "cons":  return [1.2] * 10
    elif strat == "aggr":  return [2.0] * 10
    return [1.5] * 10


def simulate_one_run(sales, archetype, rng,
                     min_floor=0.0, use_floor=False,
                     max_carryover=0.0, use_max_carry=False):
    """
    One Monte Carlo lifecycle given a drawn sales curve.

    Constraints applied each year in order:
      1. Strategy multiplier sets base planned production
      2. Max carryover cap — if carry-in already exceeds cap, skip production
         (there is sufficient inventory; no need to produce more)
      3. Minimum production floor — if producing, never produce less than floor
         (reflects minimum viable batch size)
    """
    y_draw, c_draw = sample_yield_conv_normal(archetype, rng)
    carryover = 0.0
    rows, missed_sales = [], []
    mults = get_yearly_multipliers()

    for yr in range(10):
        expected_next = sales[yr + 1] if yr < 9 else 0.0
        planned_prod  = mults[yr] * expected_next

        # Max carryover cap: skip production if carry-in already exceeds cap
        if use_max_carry and max_carryover > 0 and carryover >= max_carryover:
            planned_prod = 0.0

        # Min production floor: raise production if below floor (but only if producing)
        if use_floor and min_floor > 0 and planned_prod > 0.0:
            planned_prod = max(planned_prod, min_floor)

        new_prod       = planned_prod * y_draw * c_draw
        prod_loss      = new_prod * 0.02
        carry_loss     = carryover * 0.10
        total_saleable = (carryover - carry_loss) + (new_prod - prod_loss)
        remaining      = total_saleable - sales[yr]

        missed_sales.append(max(0.0, -remaining))
        rows.append([carryover, -carry_loss, planned_prod, new_prod,
                     -prod_loss, total_saleable, sales[yr], remaining])
        carryover = remaining

    return np.array(rows, dtype=float), np.array(missed_sales, dtype=float)


def _get_constraint_params():
    """Read current constraint widget values."""
    return {
        "use_floor":      bool(enable_min_floor_chk.value),
        "min_floor":      float(min_floor_box.value),
        "use_max_carry":  bool(enable_max_carry_chk.value),
        "max_carryover":  float(max_carry_box.value),
    }


def _determine_analysis_year(median_remaining):
    """Resolve Year Mode widget to a 0-based year index."""
    mode = year_mode_dd.value
    if mode == "last_sales":
        idx = np.where(median_remaining > -1e9)[0]
        return int(idx[-1]) if len(idx) else 9
    return int(analysis_year_dd.value)


def _run_product_mc(arch, maturity, iterations, rng, constraints):
    """Run Monte Carlo for one archetype|maturity. Returns (iters, 10, 8) array."""
    prod_rows = []
    for _ in range(int(iterations)):
        sales = draw_sales_curve(arch, maturity, rng)
        rows, _ = simulate_one_run(sales, arch, rng, **constraints)
        prod_rows.append(rows)
    return np.stack(prod_rows, axis=0)


def _prob_summary_row(arr, label, thr):
    """Build one probability summary row dict."""
    return {
        "Scenario":                        label,
        "Median remaining":                round(float(np.median(arr)), 1),
        "Mean remaining":                  round(float(arr.mean()), 1),
        "P90 remaining":                   round(float(np.percentile(arr, 90)), 1),
        f"P(remaining > {thr:.0f})":       f"{(arr > thr).mean():.1%}",
        "P(stockout)":                     f"{(arr < 0).mean():.1%}",
    }


def build_lifecycle_sim(products, iterations, seed):
    """
    Single-start portfolio Monte Carlo.
    Outputs per-product median tables, portfolio aggregate, probability summary,
    and sales validation table.
    """
    parsed = []
    for p in products:
        arch, mat_str = p.split("|")
        arch = arch.strip(); maturity = int(mat_str.strip())
        if build_sales_curve(arch, maturity) is None:
            print(f"\u274c Missing sales data for {arch} | {maturity}; skipping.")
            continue
        parsed.append((arch, maturity))

    if not parsed:
        print("\u274c No valid products selected.")
        return

    constraints = _get_constraint_params()
    rng = np.random.default_rng(int(seed))
    run_rows_all = []

    year_labels = [f"Year {i}" for i in range(1, 11)]
    row_index   = [
        "Carry-in inventory (from prior year)",
        "Carry-in quality loss (10%)",
        "Planned production",
        "Actual production (after yield & conversion)",
        "Production quality loss (2%)",
        "Total saleable inventory",
        "Sales",
        "Remaining inventory  [negative = stockout]"
    ]

    # ── Header ────────────────────────────────────────────────────────────────
    print("\n" + "=" * 66)
    strat_label = next((lbl for lbl, val in strategy_dd.options if val == strategy_dd.value), strategy_dd.value)
    print(f"  Single-start Monte Carlo  |  Strategy: {strat_label}")
    print(f"  Simulations: {iterations:,}  |  Seed: {seed}")
    print(f"  Sales variability: lognormal (salesVariability.xlsx)")
    if constraints["use_floor"]:
        print(f"  Min production floor : {constraints['min_floor']:,.0f} units  [ENABLED]")
    if constraints["use_max_carry"]:
        print(f"  Max carryover cap    : {constraints['max_carryover']:,.0f} units  [ENABLED]")
    print("=" * 66)

    # ── Per-product tables ────────────────────────────────────────────────────
    for arch, maturity in parsed:
        prod_runs = _run_product_mc(arch, maturity, iterations, rng, constraints)
        run_rows_all.append(prod_runs)

        median_rows = np.median(prod_runs, axis=0)
        df_prod = pd.DataFrame(median_rows.T, columns=year_labels, index=row_index)
        print(f"\n── {arch} | Maturity {maturity}  (median, {iterations:,} runs) ──")
        display(df_prod.round(1))

        # Per-product probability summary
        rem_prod = prod_runs[:, :, 7]
        med_rem  = np.median(rem_prod, axis=0)
        ay_prod  = _determine_analysis_year(med_rem)
        thr      = float(threshold_slider.value)
        p_rows   = [_prob_summary_row(rem_prod[:, ay_prod],
                                      f"Year {ay_prod+1} (analysis year)", thr)]
        if ay_prod != 9:
            p_rows.append(_prob_summary_row(rem_prod[:, -1], "Year 10", thr))
        display(pd.DataFrame(p_rows).style.hide(axis="index"))

    # ── Portfolio aggregate ───────────────────────────────────────────────────
    run_rows      = np.sum(run_rows_all, axis=0)
    remaining_all = run_rows[:, :, 7]

    if len(parsed) > 1:
        median_rows_agg = np.median(run_rows, axis=0)
        df_agg = pd.DataFrame(median_rows_agg.T, columns=year_labels, index=row_index)
        print(f"\n── PORTFOLIO AGGREGATE  ({len(parsed)} products, {iterations:,} runs) ──")
        display(df_agg.round(1))

        # Portfolio probability summary
        median_remaining = np.median(remaining_all, axis=0)
        ay_idx   = _determine_analysis_year(median_remaining)
        thr      = float(threshold_slider.value)
        agg_rows = [_prob_summary_row(remaining_all[:, ay_idx],
                                      f"Year {ay_idx+1} (analysis year)", thr)]
        if ay_idx != 9:
            agg_rows.append(_prob_summary_row(remaining_all[:, -1], "Year 10", thr))
        print(f"\n── Portfolio probability summary  (threshold = {thr:,.0f} units) ──")
        display(pd.DataFrame(agg_rows).style.hide(axis="index"))

    # ── Validation table ──────────────────────────────────────────────────────
    print("\n── Validation: simulated median vs data.xlsx reference ──")
    print("   Simulated median ≈ reference median confirms correct implementation.")
    print("   Simulated mean > median is expected (lognormal right skew).\n")
    val_rows = []
    for arch, maturity in parsed:
        draws   = np.array([draw_sales_curve(arch, maturity,
                            np.random.default_rng(seed + i)) for i in range(2000)])
        ref     = build_sales_curve(arch, maturity)
        sim_med = np.median(draws, axis=0)
        sim_mn  = draws.mean(axis=0)
        for i, yr in enumerate(year_labels):
            val_rows.append({
                "Product":                      f"{arch} | {maturity}",
                "Year":                         yr,
                "Reference median (data.xlsx)": round(float(ref[i]), 1),
                "Simulated median":             round(float(sim_med[i]), 1),
                "Simulated mean":               round(float(sim_mn[i]), 1),
                "Median error (%)":             round(float(abs(sim_med[i] - ref[i])
                                                / max(ref[i], 1) * 100), 1),
            })
    display(pd.DataFrame(val_rows).style.hide(axis="index"))


In [138]:
def run_multiyear_launch_sim(iterations, seed, lifecycle_years=10):
    """
    Multi-year launch Monte Carlo using launch_plan_df.
    Each cohort (arch, mat, launch_year, n_products) is simulated independently
    and time-shifted onto a shared calendar timeline.
    Outputs per-product calendar tables, portfolio aggregate, and probability summary.
    """
    rng      = np.random.default_rng(int(seed))
    cohorts  = build_launch_cohorts()
    constraints = _get_constraint_params()

    if not cohorts:
        print("\u274c No launches defined. Fill in the launch plan table above.")
        return

    max_launch = max(c[2] for c in cohorts)
    T          = max_launch + lifecycle_years   # total calendar years

    all_runs = np.zeros((int(iterations), T, 8), dtype=float)

    unique_products = sorted({(arch, mat) for (arch, mat, _, _) in cohorts})
    product_runs    = {}

    for (arch, mat) in unique_products:
        base_runs = _run_product_mc(arch, mat, iterations, rng, constraints)
        prod_all  = np.zeros((int(iterations), T, 8), dtype=float)

        for (_, _, launch_year, n_products) in [
            c for c in cohorts if c[0] == arch and c[1] == mat
        ]:
            for _ in range(n_products):
                for it in range(int(iterations)):
                    rows  = base_runs[it, :, :]
                    start = launch_year
                    end   = min(launch_year + lifecycle_years, T)
                    span  = end - start
                    prod_all[it, start:end, :] += rows[:span, :]

        product_runs[(arch, mat)] = prod_all
        all_runs += prod_all

    # ── Setup ─────────────────────────────────────────────────────────────────
    year_cols = [f"Year {i}" for i in range(1, T + 1)]
    row_index = [
        "Carry-in inventory (from prior year)",
        "Carry-in quality loss (10%)",
        "Planned production",
        "Actual production (after yield & conversion)",
        "Production quality loss (2%)",
        "Total saleable inventory",
        "Sales",
        "Remaining inventory [negative = stockout]",
    ]
    thr = float(threshold_slider.value)

    # ── Header ────────────────────────────────────────────────────────────────

    print("=" * 66)
    print("  Multi-year launch Monte Carlo")
    strat_label = next((lbl for lbl, val in strategy_dd.options if val == strategy_dd.value), strategy_dd.value)
    print(f"  Strategy     : {strat_label}")
    print(f"  Simulations  : {iterations:,}   |   Seed: {seed}")
    print(f"  Launch years : Y{LAUNCH_YEARS[0]} – Y{LAUNCH_YEARS[-1]}  "
          f"|  Horizon: {T} calendar years")
    print(f"  Sales variability: lognormal (salesVariability.xlsx)")
    if constraints["use_floor"]:
        print(f"  Min production floor : {constraints['min_floor']:,.0f} units  [ENABLED]")
    if constraints["use_max_carry"]:
        print(f"  Max carryover cap    : {constraints['max_carryover']:,.0f} units  [ENABLED]")
    print("=" * 66)

    # ── Per-product calendar tables ───────────────────────────────────────────
    for (arch, mat), runs in product_runs.items():
        median_rows_prod = np.median(runs, axis=0)
        df_prod = pd.DataFrame(median_rows_prod.T, columns=year_cols, index=row_index)
        print(f"\n── {arch} | Maturity {mat}  "
              f"(median, {iterations:,} runs | calendar years 1–{T}) ──")
        display(df_prod.round(1))

        # Per-product probability summary
        rem_prod     = runs[:, :, 7]
        med_rem_prod = np.median(rem_prod, axis=0)
        sales_prod   = df_prod.loc["Sales"].astype(float).values
        s_idx        = np.where(sales_prod > 0)[0]
        ay_prod      = int(s_idx[-1]) if len(s_idx) else T - 1
        p_rows_prod  = [_prob_summary_row(rem_prod[:, ay_prod],
                                          f"Year {ay_prod+1} (last sales year)", thr)]
        p_rows_prod.append(_prob_summary_row(rem_prod[:, -1],
                                             f"End of horizon (Year {T})", thr))
        display(pd.DataFrame(p_rows_prod).style.hide(axis="index"))

    # ── Portfolio aggregate ───────────────────────────────────────────────────
    median_rows_port = np.median(all_runs, axis=0)
    lifecycle_df     = pd.DataFrame(median_rows_port.T, columns=year_cols, index=row_index)

    print(f"\n── PORTFOLIO AGGREGATE  "
          f"({len(unique_products)} archetypes, {iterations:,} runs | "
          f"calendar years 1–{T}) ──")
    display(lifecycle_df.round(1))

    # ── Portfolio probability summary ─────────────────────────────────────────
    remaining_all = all_runs[:, :, 7]
    sales_row     = lifecycle_df.loc["Sales"].astype(float).values
    s_idx         = np.where(sales_row > 0)[0]
    ay_idx        = int(s_idx[-1]) if len(s_idx) else T - 1

    agg_rows = [_prob_summary_row(remaining_all[:, ay_idx],
                                  f"Year {ay_idx+1} (last sales year)", thr)]
    agg_rows.append(_prob_summary_row(remaining_all[:, -1],
                                      f"End of horizon (Year {T})", thr))
    print(f"\n── Portfolio probability summary  (threshold = {thr:,.0f} units) ──")
    display(pd.DataFrame(agg_rows).style.hide(axis="index"))

    # ── Sanity check ──────────────────────────────────────────────────────────
    total_sales = lifecycle_df.loc["Sales"].sum()
    print(f"\n── Sanity check ──")
    print(f"   Total portfolio sales over {T}-year horizon : {total_sales:,.0f} units (median)")
    print(f"   Sales should taper off after Year {max_launch + lifecycle_years}.")
    print(f"   Remaining inventory should approach zero by Year {T}.\n")


**Portfolio simulation, output tables, chart, and validation**

This block defines:

- `simulate_one_run()` — one lifecycle with stochastic yield/conversion.
  Calls `draw_sales_curve()` (defined in the lookup cell) inside the loop
  so each run gets its own lognormal sales trajectory.
- `build_lifecycle_sim()` — full Monte Carlo runner that:
  1. Shows a **median lifecycle table per product** (median across all runs —
     not inflated by right-tail outliers unlike the mean)
  2. Shows a **portfolio aggregate table** when multiple products are selected
  3. Shows a **Plotly chart** with median, mean (dotted), and P10–P90 band for
     remaining inventory so the mean/median gap is visually clear
  4. Shows a **probability summary** (P stockout, P > threshold)
  5. Shows a **validation table** comparing simulated median against the
     data.xlsx reference median — if the implementation is correct these
     should match within ~2%; the mean being higher than the median is
     expected and correct due to lognormal right skew


In [139]:
mode_radio = widgets.RadioButtons(
    options=[
        ("Single-start portfolio (current)", "single"),
        ("Multi-year launch cohorts", "multi"),
    ],
    value="single",
    description="Mode:",
)


In [140]:
def on_mode_change(change):
    if change["name"] == "value":
        if change["new"] == "multi":
            multi_year_box.layout.display = "block"
        else:
            multi_year_box.layout.display = "none"

mode_radio.observe(on_mode_change, names="value")


In [141]:
run_button = widgets.Button(
    description="Run simulation",
    button_style="primary",
    layout=widgets.Layout(width="200px")
)

output_area    = widgets.Output()
selection_label = widgets.HTML(value="")

def on_run_clicked(b):
    output_area.clear_output()
    with output_area:
        if mode_radio.value == "single":
            build_lifecycle_sim(
                tuple(products_ms.value),
                int(iters_slider.value),
                int(seed_box.value),
            )
        else:
            sync_launch_plan_df()
            run_multiyear_launch_sim(
                iterations=int(iters_slider.value),
                seed=int(seed_box.value),
                lifecycle_years=10,
            )

run_button.on_click(on_run_clicked)

SEP = widgets.HTML("<hr style='margin:6px 0;border-color:#ddd'>")

dashboard = widgets.VBox([
    widgets.HTML("<h3 style='margin:4px 0'>GDM Monte Carlo Inventory Planner</h3>"),

    widgets.HTML("<b>Simulation mode</b>"),
    mode_radio,
    multi_year_box,

    SEP,
    widgets.HTML("<b>Product selection</b> "
                 "<small style='color:#666'>(Ctrl+click or Cmd+click to select multiple)</small>"),
    products_ms,

    SEP,
    widgets.HTML("<b>Production strategy</b> "
                 "<small style='color:#666'>(multiplier applied to next year's expected sales)</small>"),
    strategy_dd,
    widgets.VBox(year_mult_sliders),

    SEP,
    widgets.HTML("<b>Simulation settings</b>"),
    widgets.HBox([iters_slider, seed_box]),

    SEP,
    widgets.HTML("<b>Analysis year</b> "
                 "<small style='color:#666'>(which year to show probability stats for)</small>"),
    widgets.HBox([year_mode_dd, analysis_year_dd]),
    widgets.HTML("<b>Carryover threshold</b> "
                 "<small style='color:#666'>"
                 "P(remaining inventory exceeds this value) is reported for each product and portfolio"
                 "</small>"),
    threshold_slider,

    SEP,
    widgets.HTML("<b>Production constraints</b>"),
    widgets.HTML(
        "<small style='color:#555'>"
        "<b>Min production floor</b> — if producing in a year, never plan below this amount "
        "(reflects minimum viable batch size). Does not force production in zero-production years."
        "<br>"
        "<b>Max carryover cap</b> — if carry-in inventory already meets or exceeds this level, "
        "skip production that year (sufficient stock on hand)."
        "</small>"
    ),
    widgets.HBox([enable_min_floor_chk, min_floor_box]),
    widgets.HBox([enable_max_carry_chk, max_carry_box]),

    SEP,
    run_button,
    widgets.HTML("<br>"),
    output_area,
])

display(dashboard)
